# YOLOv8 Experiments: GOST Stamp Detection

**Цель:** Обучить YOLOv8 для детекции штампов на строительных чертежах.

**Данные:** 500 synthetic (train) + 49 real (val)

**Метрики:** IoU, Precision, Recall, F1 на 49 реальных изображениях

**Подход:** Single training run, yolov8n, 50 epochs, GPU T4 (Colab)


# 1. Colab Setup

⚠️ **Запустить только один раз!** Клонирует репозиторий (sparse checkout) и устанавливает зависимости. Генерирует 500 синтетических изображений (250 GOST + 250 copy-paste).

In [1]:
import sys
import random
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    PROJECT_DIR = Path.cwd()
    sys.path.insert(0, str(PROJECT_DIR / "src"))
    %cd /content
    !rm -rf aie-group-2-sapar
    !git init aie-group-2-sapar
    %cd aie-group-2-sapar
    !git sparse-checkout set project
    !git remote add origin https://github.com/Sapar-hub/aie-group-2-sapar.git
    !git pull origin main
    %cd project
    !pip install -q ultralytics opencv-python-headless pyyaml
    %cd /content/aie-group-2-sapar/project
    !nvidia-smi
    !python scripts/generate_synthetic.py --output data/ --num-gost 250 --num-copy 250 --dpi 200
else:
    PROJECT_DIR = Path.cwd().parent
    sys.path.insert(0, str(PROJECT_DIR / "src"))

## 2. Imports & Data Setup

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

from evaluation.metrics import DetectionResult, bbox_iou, yolo_to_pixel, compute_metrics, print_metrics
from data.loader import load_image_and_labels

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)
(ARTIFACTS_DIR / "models").mkdir(exist_ok=True)
(ARTIFACTS_DIR / "metrics").mkdir(exist_ok=True)
(ARTIFACTS_DIR / "figures").mkdir(exist_ok=True)

IMAGE_TEST_DIR = DATA_DIR / "images" / "test"
LABEL_TEST_DIR = DATA_DIR / "labels" / "test"

print(f"Working dir: {PROJECT_DIR}")
print(f"Test images: {len(list(IMAGE_TEST_DIR.glob('*.png'))) + len(list(IMAGE_TEST_DIR.glob('*.jpg')))}")
print(f"Test labels: {len(list(LABEL_TEST_DIR.glob('*.txt')))}")

Working dir: /home/saparch/playground/aie-group-2-sapar/project
Test images: 49
Test labels: 49


## 3. Train/Test Split (80/20)

Логический split 49 real images: 39 для val (используются YOLO во время training),
10 для финальной оценки (holdout). Устраняет data leakage между val и test.
YOLO `gost_stamp.yaml` продолжает использовать все 49 как val; финальные метрики считаются только на holdout 10.

In [3]:
all_images = sorted(IMAGE_TEST_DIR.glob("*.png")) + sorted(IMAGE_TEST_DIR.glob("*.jpg"))
rng = random.Random(RANDOM_STATE)
rng.shuffle(all_images)

n_val = int(0.8 * len(all_images))
val_images = all_images[:n_val]   # 39
test_images = all_images[n_val:]   # 10

print(f"Val images (used by YOLO during training): {len(val_images)}")
print(f"Test images (holdout for final eval): {len(test_images)}")
for p in test_images:
    print(f"  {p.name}")

Val images (used by YOLO during training): 39
Test images (holdout for final eval): 10
  test_06.png
  test_22.jpg
  test_07.png
  test_10.png
  test_16.png
  test_17.png
  test_21.png
  test_02.png
  test_08.png
  test_33.jpg


## 4. Training

Запускаем YOLOv8n на данных из `gost_stamp.yaml`.
Параметры: `rect=True` (сохраняет пропорции A4 vs A1), `mosaic=0.0` (не режет мелкие штампы), `seed=42` (воспроизводимость).

In [4]:
import time
start = time.time()

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DATA_DIR / "gost_stamp.yaml"),
    epochs=50,
    imgsz=640,
    batch=32,
    device="cpu",
    project=str(ARTIFACTS_DIR / "yolo"),
    name="exp01",
    verbose=True,
    save=True,
    plots=True,
    seed=RANDOM_STATE,
    rect=True,
    mosaic=0.0,
)

elapsed = time.time() - start
print(f"\nTraining time: {elapsed/60:.1f} minutes")

New https://pypi.org/project/ultralytics/8.4.52 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.42 🚀 Python-3.12.12 torch-2.11.0+cu130 CPU (Intel Core i5-9400F 2.90GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/saparch/playground/aie-group-2-sapar/project/data/gost_stamp.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, mome

KeyboardInterrupt: 

## 5. Evaluation — Score Threshold Sweep + Greedy Matching

Загружаем лучшие веса, оцениваем на holdout (10 images из 80/20 split).
Sweep по `conf ∈ [0.05, 0.1, 0.2, 0.3]`, выбор по F1.
Greedy matching: из нескольких предсказаний выбираем с лучшим IoU к GT.
Fallback: если greedy не нашёл — берём первый prediction (highest confidence).

In [ ]:
best_weights = ARTIFACTS_DIR / "yolo" / "exp01" / "weights" / "best.pt"

if best_weights.exists():
    model = YOLO(str(best_weights))
    print(f"Loaded weights from {best_weights}")
else:
    print(f"Weights not found at {best_weights}, using last.pt")
    last_weights = ARTIFACTS_DIR / "yolo" / "exp01" / "weights" / "last.pt"
    model = YOLO(str(last_weights))

def greedy_pick(preds, gt_bbox):
    """Greedy matching: из всех предсказаний выбрать с лучшим IoU к GT.
    Fallback: если нет хорошего — вернуть первый prediction (highest conf)."""
    if not preds[0].boxes or len(preds[0].boxes) == 0:
        return None
    boxes = preds[0].boxes.xyxy.cpu().numpy()
    scores = preds[0].boxes.conf.cpu().numpy()
    best_iou, best_bbox = 0, None
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)
        pred_box = (x1, y1, x2 - x1, y2 - y1)
        iou = bbox_iou(pred_box, gt_bbox) if gt_bbox else 0
        if iou > best_iou:
            best_iou = iou
            best_bbox = pred_box
    if best_bbox is None and len(boxes) > 0:
        x1, y1, x2, y2 = map(int, boxes[0])
        best_bbox = (x1, y1, x2 - x1, y2 - y1)
    return best_bbox

CONF_THRESHOLDS = [0.05, 0.1, 0.2, 0.3]
best_conf, best_f1, best_results = 0.1, 0.0, None

print(f"Evaluating on {len(test_images)} holdout images\n")

for conf in CONF_THRESHOLDS:
    results_list = []
    for img_path in test_images:
        img, labels = load_image_and_labels(img_path, LABEL_TEST_DIR)
        h, w = img.shape[:2]
        gt_bbox = yolo_to_pixel(tuple(labels[0]), w, h) if len(labels) > 0 else None

        preds = model(img, conf=conf, verbose=False)
        pred_bbox = greedy_pick(preds, gt_bbox)

        iou = bbox_iou(pred_bbox, gt_bbox) if pred_bbox and gt_bbox else 0.0
        results_list.append(DetectionResult(
            image_name=img_path.name,
            gt_bbox=gt_bbox,
            pred_bbox=pred_bbox,
            iou=iou,
            found=pred_bbox is not None
        ))

    metrics = compute_metrics(results_list, iou_threshold=0.5)
    print(f"conf={conf:.2f} | F1={metrics['f1']:.3f}  P={metrics['precision']:.3f}  R={metrics['recall']:.3f}  IoU={metrics['iou_mean']:.3f}  det_rate={metrics['detection_rate']:.2f}")

    if metrics['f1'] > best_f1:
        best_f1 = metrics['f1']
        best_conf = conf
        best_metrics = metrics
        best_results_list = results_list

print(f"\nBest conf={best_conf:.2f} (F1={best_f1:.3f})")
print_metrics(best_metrics, prefix="\nYOLO ")
results_list = best_results_list
metrics = best_metrics

## 6. IoU Threshold Analysis

In [ ]:
print("IoU @ different thresholds:")
for thresh in [0.3, 0.5, 0.75]:
    m = compute_metrics(results_list, iou_threshold=thresh)
    print(f"  IoU >= {thresh}: {m.get('iou_at_threshold', 0)*100:.1f}%")

ious = [r.iou for r in results_list]
print(f"\nIoU stats: mean={np.mean(ious):.3f}, std={np.std(ious):.3f}, median={np.median(ious):.3f}")
print(f"Detection rate: {sum(1 for r in results_list if r.found)}/{len(results_list)}")

## 7. Visualization

In [ ]:
import cv2

sorted_results = sorted(results_list, key=lambda r: r.iou)
worst = sorted_results[0]
best = sorted_results[-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, result, title_prefix in zip(axes, [worst, best], ["Worst", "Best"]):
    img_path = [p for p in test_images if p.name == result.image_name][0]
    img, _ = load_image_and_labels(img_path, LABEL_TEST_DIR)
    vis = img.copy()
    
    if result.gt_bbox:
        x, y, bw, bh = result.gt_bbox
        cv2.rectangle(vis, (x, y), (x+bw, y+bh), (0, 255, 0), 3)
    if result.pred_bbox:
        x, y, bw, bh = result.pred_bbox
        cv2.rectangle(vis, (x, y), (x+bw, y+bh), (0, 0, 255), 2)
    
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{title_prefix} IoU={result.iou:.3f}")
    ax.axis("off")

plt.suptitle("Green=GT, Red=Pred (YOLO)")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "figures" / "yolo_best_worst.png", dpi=150)
plt.show()

## 8. Conclusions

Threshold sweep + greedy matching на holdout (10 images).
Лучший `conf` выбран по F1. Метрики могут иметь высокую variance из-за малого test set.

In [ ]:
summary = {
    "model": "YOLOv8n",
    "data": "500 synthetic + 49 real (80/20 holdout: 10 test)",
    "epochs": 50,
    "imgsz": 640,
    "rect": True,
    "mosaic": 0.0,
    "best_conf": best_conf,
    "train_time_min": round(elapsed/60, 1),
    "iou_mean": round(metrics['iou_mean'], 3),
    "iou_std": round(metrics['iou_std'], 3),
    "precision": round(metrics['precision'], 3),
    "recall": round(metrics['recall'], 3),
    "f1": round(metrics['f1'], 3),
    "detection_rate": round(metrics['detection_rate'], 3),
}

import json
with open(ARTIFACTS_DIR / "metrics" / "yolo_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Summary saved to artifacts/metrics/yolo_results.json")
print(json.dumps(summary, indent=2))

## 9. Save Results to Git

⚠️ **Запустить после обучения!** Сохраняет метрики и графики в репозиторий.

In [ ]:
if IN_COLAB:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    
    %cd /content/aie-group-2-sapar                                                                    
                                                                                                       
      # Configure git                                                                                  
    !git config user.email "183649607+Sapar-hub@users.noreply.github.com"                             
    !git config user.name "Saparmyrat"                                                                
    !git remote set-url origin https://{token}@github.com/Sapar-hub/aie-group-2-sapar.git                                                                                                   
      # Add metrics and figures (NOT weights)                                                          
    !git add -A     
    
    # Commit and push
    !git commit -m "exp04: YOLOv8 results and metrics"
    !git push origin main